### Factor Decomposition for an S&P Portfolio

In [4]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src import data, factors, plotting

spy_px, spy_ret = data.get_market()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
plotting.plot_stock(prices, "AAPL")
plotting.plot_vs_market(prices, spy_px, "AAPL")

### Single-Factor Model: A Naive Look at Idiosyncratic Returns

We decompose a stock's daily return into a market component and a stock-specific component:

$$
r_{i,t} = \alpha_i + \beta_i \, r_{m,t} + \varepsilon_{i,t}
$$

- $r_{i,t}$: return of stock $i$ on day $t$
- $r_{m,t}$: market (SPY) return on day $t$
- $\beta_i = \dfrac{\mathrm{Cov}(r_i, r_m)}{\mathrm{Var}(r_m)}$: sensitivity to the market
- $\alpha_i$: average return not explained by the market
- $\varepsilon_{i,t}$: residual, the **idiosyncratic return**

The idiosyncratic return is what's left after stripping out the market:

$$
\varepsilon_{i,t} = r_{i,t} - \alpha_i - \beta_i \, r_{m,t}
$$

This also splits total variance into systematic and idiosyncratic parts (the residual is uncorrelated with the market by construction):

$$
\mathrm{Var}(r_i) = \beta_i^2 \, \mathrm{Var}(r_m) + \mathrm{Var}(\varepsilon_i)
$$

**Why it's naive:** the market is the only factor. Anything driven by sector or style (size, momentum, volatility) gets lumped into $\varepsilon$, so this overstates true idiosyncratic risk. A multi-factor model peels those layers off next.

In [12]:
plotting.plot_regression(prices, spy_px, "AAPL")

In [13]:
qqq_px, qqq_ret = data.get_market("QQQ")
plotting.plot_regression(prices, qqq_px, "AAPL", market_name="QQQ")

In [14]:
iwm_px, _ = data.get_market("IWM")   # Russell 2000 (small caps)
plotting.plot_regression(prices, iwm_px, "AAPL", market_name="IWM")

### Two-Factor Model: Market + Sector

Extend the single-factor model with a sector return (e.g. XLK for Apple):

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, r_{s,t} + \varepsilon_{i,t}
$$

**Problem: collinearity.** Sector and market returns are highly correlated ($\rho \approx 0.85$ to $0.9$ for XLK vs SPY), so the two betas are unstable and hard to interpret.

**Fix: orthogonalize the sector against the market.** First strip the market out of the sector:

$$
r_{s,t} = a + b \, r_{m,t} + u_t
$$

Then regress the stock on the market and the sector residual $u_t$:

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, u_t + \varepsilon_{i,t}
$$

- $u_t$: the sector-specific move, i.e. what the sector did beyond the market
- $\beta_m$: identical to the single-factor beta, since $u$ is uncorrelated with $r_m$
- $\beta_s$: sensitivity to the sector-specific move
- $\Delta R^2$ vs the single-factor model: variance explained by the sector on top of the market

**Caveat: self-inclusion.** Apple is a large weight in XLK, so part of the "sector" is Apple itself, which inflates $\beta_s$ and $R^2$. Cleaner proxies are an equal-weight sector ETF or the average return of the other stocks in the sector, excluding the one being modeled.

___

### Orthogonalization: The Idea

Treat each return series as a vector, one entry per day. Two series that move together point in similar directions. That overlap is what makes their effects hard to separate in a regression.

Orthogonalizing $s$ against $m$ splits $s$ into two pieces:

$$
s = \underbrace{b \, m}_{\text{part explained by } m} + \underbrace{u}_{\text{part unrelated to } m}
$$

- $b \, m$ is the **projection** of $s$ onto $m$: the "shadow" $s$ casts in the market's direction, with $b = \dfrac{\mathrm{Cov}(s, m)}{\mathrm{Var}(m)}$
- $u = s - b \, m$ is what's left, and by construction $\mathrm{Cov}(u, m) = 0$

Geometrically, $u$ is perpendicular to $m$: it carries no information about the market at all.

**Why it helps:** the two regressors $m$ and $u$ no longer overlap, so each coefficient is cleanly identified. The market beta is unchanged from the single-factor model, and the sector coefficient captures only what's new, i.e. the sector's own move with the market stripped out.

**The catch:** the result depends on order. Orthogonalizing the sector against the market gives the market "first claim" on the shared variance. Reversing the order would credit that shared variance to the sector instead.
___

### Adding More Factors: Sequential Orthogonalization (Gram-Schmidt)

To add a third factor that may overlap with the first two (e.g. a size proxy like IWM), regress it on **all existing factors at once** and keep the residual:

$$
r_{\text{new},t} = c + d_1 \, r_{m,t} + d_2 \, u_t + v_t
$$

Then add $v_t$ to the stock regression:

$$
r_{i,t} = \alpha_i + \beta_m \, r_{m,t} + \beta_s \, u_t + \beta_v \, v_t + \varepsilon_{i,t}
$$

- $v_t$: the part of the new factor unexplained by market and sector
- $\beta_m, \beta_s$: unchanged from the two-factor model, since $v$ is uncorrelated with both
- $\beta_v$: sensitivity to what's genuinely new in the third factor
- $\Delta R^2$: extra variance explained by the new factor

Because $m$ and $u$ are already orthogonal, regressing on them jointly or one at a time gives the same result. With correlated factors, only the joint regression gives a clean residual.

### Key point: orthogonalization changes interpretation, not fit

Regressing on raw SPY, XLK and IWM together gives the **same** $\varepsilon$ and **same** $R^2$ as the orthogonalized version, since both span the same space. What changes is **who gets credit for shared variance**:

1. Market gets first claim
2. Sector gets what's left beyond market
3. New factor gets what's left beyond both

Order is therefore a modeling choice: put the most fundamental factor first, and each later factor is measured as an increment.

- **If you care about the idiosyncratic residual**, orthogonalization doesn't matter.
- **If you care about clean, stable, interpretable betas**, it does.

In [17]:
xlk_px, _ = data.get_market("XLK")
plotting.plot_vs_market(prices, xlk_px, "AAPL", market_name="XLK")

In [18]:
plotting.plot_two_factor(prices, spy_px, xlk_px, "AAPL")

In [22]:
# market only
plotting.plot_actual_vs_fitted(prices, {"SPY": spy_px}, "AAPL")

In [23]:
# market, then XLK orthogonalized to market
plotting.plot_actual_vs_fitted(prices, {"SPY": spy_px, "XLK": xlk_px}, "AAPL")

___

### Simple 12-Month Momentum
___

In [26]:
mom = factors.momentum(prices)
z_mom = factors.standardize(mom)
res = factors.cs_regression(returns, z_mom)

plotting.plot_factor_returns(res["factor_ret"])

In [25]:
plotting.plot_cs_r2(res["r2"])

___

### Simple Volatility Factor
___

In [29]:
import numpy as np

vol = factors.volatility(returns)
z_vol = factors.standardize(np.log(vol))
res_vol = factors.cs_regression(returns, z_vol)

plotting.plot_factor_returns(res_vol["factor_ret"], name="Volatility (63d)")

In [30]:
plotting.plot_cs_r2(res_vol["r2"], name="Volatility (63d)")

In [32]:
plotting.plot_exposure_corr(z_mom, z_vol)

In [40]:
import pandas as pd

res_joint = factors.cs_regression_multi(returns, {"Momentum": z_mom, "Volatility": z_vol})

plotting.plot_factor_compare(
    {"Momentum (isolated)": res["factor_ret"], "Momentum (joint)": res_joint["Momentum"]},
    title="Momentum: isolated vs controlling for volatility",
)

In [41]:
plotting.plot_factor_compare(
    {"Volatility (isolated)": res_vol["factor_ret"], "Volatility (joint)": res_joint["Volatility"]},
    title="Volatility: isolated vs controlling for momentum",
)

___

### Simple Short-Term Reversal
___

In [42]:
rev = factors.reversal(prices)
z_rev = factors.standardize(rev)
res_rev = factors.cs_regression(returns, z_rev)

plotting.plot_factor_returns(res_rev["factor_ret"], name="Reversal (1m)")

In [43]:
plotting.plot_cs_r2(res_rev["r2"], name="Reversal (1m)")

In [45]:
res_all = factors.cs_regression_multi(
    returns, {"Momentum": z_mom, "Volatility": z_vol, "Reversal": z_rev}
)

plotting.plot_factor_compare(
    {
        "Momentum (isolated)": res["factor_ret"],
        "Momentum | Vol": res_joint["Momentum"],
        "Momentum | Vol + Rev": res_all["Momentum"],
    },
    title="Momentum: isolated vs controlling for other factors",
)

In [48]:
r2 = pd.DataFrame({
    "Momentum only": res["r2"],
    "Mom + Vol": res_joint["r2"],
    "Mom + Vol + Rev": res_all["r2"],
}).agg(["mean", "median"]).mul(100).T
r2.columns = ["Mean R² (%)", "Median R² (%)"]

plotting.plot_table(
    r2, title="Cross-sectional R² by model", index_name="Model",
    caption="Daily cross-sectional regressions of stock returns on lagged standardized exposures.",
)

___

### Simple Beta
___

In [49]:
beta = factors.beta_exposure(returns, spy_ret)
z_beta = factors.standardize(beta)
res_beta = factors.cs_regression(returns, z_beta)

plotting.plot_factor_returns(res_beta["factor_ret"], name="Beta (252d)")

In [50]:
plotting.plot_cs_r2(res_beta["r2"], name="Beta (252d)")

In [51]:
plotting.plot_exposure_corr(z_beta, z_vol, "Beta", "Volatility")

In [52]:
res_4 = factors.cs_regression_multi(
    returns, {"Momentum": z_mom, "Volatility": z_vol, "Reversal": z_rev, "Beta": z_beta}
)

plotting.plot_factor_compare(
    {"Volatility (isolated)": res_vol["factor_ret"],
     "Volatility | Mom + Rev": res_all["Volatility"],
     "Volatility | Mom + Rev + Beta": res_4["Volatility"]},
    title="Volatility: how much survives controlling for beta?",
)

In [54]:
vol_versions = pd.concat({
    "Volatility (isolated)": res_vol["factor_ret"],
    "Volatility | Mom + Rev": res_all["Volatility"],
    "Volatility | Mom + Rev + Beta": res_4["Volatility"],
}, axis=1, join="inner")

plotting.plot_table(
    factors.factor_stats(vol_versions),
    title="Volatility factor: isolated vs controlling for other factors",
    subtitle=f"{vol_versions.index.min():%b %Y} to {vol_versions.index.max():%b %Y}, common dates",
    index_name="Version",
    fmt={"Days": "{:,.0f}"},
    caption="Daily cross-sectional regressions on lagged standardized exposures; returns per 1σ of exposure.",
)